# Forecasting Data Preparation

This notebook prepares the feature-engineered bilateral trade datasets for one-month-ahead forecasting across South Korea, Slovakia, and the USA. It validates the data, retains series with sufficient historical observations, and establishes a chronological train/test split.

Observations through February 2026 are used for training, while March 2026 is reserved for out-of-sample evaluation. The resulting datasets are saved separately for consistent use across all forecasting models.

## 1. Load Forecasting Data

Load the feature-engineered datasets produced by `07_feature_engineering.ipynb`. These datasets contain the selected trade series, historical observations, and engineered features for South Korea, Slovakia, and the USA.

In [1]:
from pathlib import Path

import pandas as pd

# Resolve project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURES_DIR = PROJECT_ROOT / "Data" / "features"

DATA_FILES = {
    "South Korea": "korea_trade_features.parquet",
    "Slovakia": "slovakia_trade_features.parquet",
    "USA": "usa_trade_features.parquet",
}

# Forecasting target and series identifiers
TARGET = "Value"

SERIES_KEYS = [
    "Country1",
    "Country2",
    "ProductCode",
    "TradeFlow",
]

# Load datasets
forecasting_data = {
    country: pd.read_parquet(FEATURES_DIR / filename)
    for country, filename in DATA_FILES.items()
}

for country, df in forecasting_data.items():
    df["Month"] = pd.to_datetime(df["Month"])
    print(
        f"{country}: {len(df):,} observations, "
        f"{df[SERIES_KEYS].drop_duplicates().shape[0]:,} series"
    )

South Korea: 2,455,949 observations, 27,831 series
Slovakia: 2,848,291 observations, 21,976 series
USA: 3,952,604 observations, 35,852 series


## 2. Validate the Loaded Data

Check that each dataset contains the required columns, has unique series-month observations, is chronologically ordered within each series, and contains consecutive monthly observations. These checks establish the data's structural integrity before applying eligibility filters and creating the forecasting split.

In [2]:
REQUIRED_COLUMNS = [
    "Month",
    *SERIES_KEYS,
    TARGET,
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_mean_12",
    "rolling_std_3",
    "rolling_std_6",
    "rolling_std_12",
    "month_of_year",
    "month_sin",
    "month_cos",
]

for country, df in forecasting_data.items():
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
    if missing_columns:
        raise ValueError(f"{country}: missing columns {missing_columns}")

    if df[["Month", *SERIES_KEYS]].isna().any().any():
        raise ValueError(f"{country}: missing dates or series identifiers")

    if df.duplicated(SERIES_KEYS + ["Month"]).any():
        raise ValueError(f"{country}: duplicate series-month observations")

    if df[TARGET].isna().any():
        raise ValueError(f"{country}: missing target values")

    df = df.sort_values(SERIES_KEYS + ["Month"])
    month_differences = (
        df.groupby(SERIES_KEYS, sort=False)["Month"]
        .diff()
        .dropna()
    )

    if not month_differences.map(
        lambda delta: delta.days in (28, 29, 30, 31)
    ).all():
        raise ValueError(f"{country}: non-monthly intervals detected")

    print(f"{country}: validation passed")

South Korea: validation passed
Slovakia: validation passed
USA: validation passed


## 3. Filter Eligible Series

Retain trade series with at least 36 monthly observations from the available history through February 2026. This ensures that every eligible series has a minimum of three years of historical data available before the forecast target month, March 2026.

In [3]:
CUTOFF_MONTH = pd.Timestamp("2026-03-01")
MIN_TRAINING_OBSERVATIONS = 36

eligible_forecasting_data = {}
eligibility_report = []

for country, df in forecasting_data.items():
    # Use only observations available before the target month
    historical_data = df[df["Month"] < CUTOFF_MONTH]

    # Count historical observations for each series
    series_lengths = (
        historical_data.groupby(SERIES_KEYS, sort=False)
        .size()
        .rename("n_observations")
        .reset_index()
    )

    eligible_keys = series_lengths.loc[
        series_lengths["n_observations"] >= MIN_TRAINING_OBSERVATIONS,
        SERIES_KEYS,
    ]

    # Retain eligible series, including their target-month row if available
    filtered_df = df.merge(
        eligible_keys,
        on=SERIES_KEYS,
        how="inner",
        validate="many_to_one",
    )

    eligible_forecasting_data[country] = filtered_df

    eligibility_report.append({
        "Country": country,
        "Series before": df[SERIES_KEYS].drop_duplicates().shape[0],
        "Eligible series": eligible_keys.shape[0],
        "Observations retained": len(filtered_df),
    })

display(pd.DataFrame(eligibility_report))

,Country,Series before,Eligible series,Observations retained
0,South Korea,27831,27823,2455653
1,Slovakia,21976,21973,2848183
2,USA,35852,35843,3952273


## 4. Define the Forecasting Experiment

The objective is to predict monthly trade values one month ahead. February 2026 is the final month available for model training, and March 2026 is reserved for out-of-sample evaluation.

The same forecast horizon and target month are used for South Korea, Slovakia, and the USA to ensure consistent model comparisons.

In [4]:
# Forecasting configuration
FORECAST_HORIZON = 1
TRAIN_END = pd.Timestamp("2026-02-01")
TEST_MONTH = pd.Timestamp("2026-03-01")

assert TEST_MONTH == TRAIN_END + pd.DateOffset(
    months=FORECAST_HORIZON
)

print(f"Forecast horizon: {FORECAST_HORIZON} month")
print(f"Training data through: {TRAIN_END:%B %Y}")
print(f"Evaluation month: {TEST_MONTH:%B %Y}")

Forecast horizon: 1 month
Training data through: February 2026
Evaluation month: March 2026


## 5. Create Training and Test Datasets

Split each eligible series chronologically. Observations through February 2026 are available for training, while March 2026 is held out for evaluation. The target-month value is stored separately and must never be used to fit a model or construct predictors.

In [6]:
training_data = {}
test_data = {}

for country, df in eligible_forecasting_data.items():
    training_data[country] = (
        df.loc[df["Month"] <= TRAIN_END]
        .copy()
        .reset_index(drop=True)
    )

    test_data[country] = (
        df.loc[df["Month"] == TEST_MONTH, SERIES_KEYS + ["Month", TARGET]]
        .copy()
        .reset_index(drop=True)
    )

    print(
        f"{country}: "
        f"{len(training_data[country]):,} training observations, "
        f"{len(test_data[country]):,} test observations, "
        f"{test_data[country][SERIES_KEYS].drop_duplicates().shape[0]:,} test series"
    )

South Korea: 2,374,832 training observations, 27,823 test observations, 27,823 test series
Slovakia: 2,826,210 training observations, 21,973 test observations, 21,973 test series
USA: 3,880,587 training observations, 35,843 test observations, 35,843 test series


## 6. Validate the Train/Test Split

Verify that training observations end no later than February 2026, test observations belong exclusively to March 2026, and every eligible series has exactly one test target. These checks confirm the chronological split is consistent across all three countries.

In [7]:
for country in training_data:
    train = training_data[country]
    test = test_data[country]
    eligible = eligible_forecasting_data[country]

    # Training must not contain target-month observations
    assert train["Month"].max() <= TRAIN_END
    assert not (train["Month"] >= TEST_MONTH).any()

    # Test data must contain only the target month
    assert test["Month"].eq(TEST_MONTH).all()

    # Every eligible series must have exactly one target observation
    eligible_keys = eligible[SERIES_KEYS].drop_duplicates()
    test_counts = test.groupby(SERIES_KEYS, dropna=False).size()

    assert len(test) == len(test_counts), (
        f"{country}: duplicate test targets detected"
    )
    assert len(test_counts) == len(eligible_keys), (
        f"{country}: some eligible series lack a test target"
    )
    assert test_counts.eq(1).all(), (
        f"{country}: expected exactly one target per series"
    )

    # Target values must be present
    assert test[TARGET].notna().all()

    print(f"{country}: train/test validation passed")

South Korea: train/test validation passed
Slovakia: train/test validation passed
USA: train/test validation passed


## 7. Save the Forecasting Datasets

Save the training observations and held-out March 2026 targets separately for each country. These files will serve as consistent inputs for the baseline, classical time-series, and machine-learning models.

In [9]:
OUTPUT_DIR = PROJECT_ROOT / "Data" / "forecasting"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for country, train in training_data.items():
    prefix = {
        "South Korea": "korea",
        "Slovakia": "slovakia",
        "USA": "usa",
    }[country]

    test = test_data[country]

    train_path = OUTPUT_DIR / f"{prefix}_train.parquet"
    test_path = OUTPUT_DIR / f"{prefix}_test.parquet"

    train.to_parquet(train_path, index=False)
    test.to_parquet(test_path, index=False)

    print(
        f"{country}: saved {len(train):,} training rows and "
        f"{len(test):,} test targets"
    )

South Korea: saved 2,374,832 training rows and 27,823 test targets
Slovakia: saved 2,826,210 training rows and 21,973 test targets
USA: saved 3,880,587 training rows and 35,843 test targets


## 8. Conclusion

This notebook prepares the data for one-month-ahead forecasting of bilateral trade flows across South Korea, Slovakia, and the USA. Series eligibility is determined using historical observations available before March 2026, and the training/test split reserves March 2026 for out-of-sample evaluation.

The training datasets and held-out target values are saved separately in `data/forecasting/` for consistent reuse across forecasting models.

The prepared datasets are ready for baseline forecasting and subsequent model benchmarking.